# PV Systematic v4 — Full Pipeline + ZoLa + QA + Auto-Export
Run Stages 1–10.

### Imports and Config


In [ ]:
import os
import requests
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import Polygon, box
import subprocess
from folium.plugins import MarkerCluster


## Stage 1: Define Bounding Box (DATA SOURCE)

In [ ]:
# import importlib
# import scripts.fetch_selection as s1
# importlib.reload(s1); s1.main()

N_LAT = 40.7358876957036
S_LAT = 40.71195687391828
W_LON = -74.00188877257987
E_LON = -73.9721415592451

FOOTPRINTS_URL = "https://data.cityofnewyork.us/resource/5zhs-2jue.geojson"

params = {
    "$select": "*",
    "$where": f"within_box(the_geom,{S_LAT},{W_LON},{N_LAT},{E_LON})",
    "$limit": 50000,
}

## Stage 2: Fetch + Save Footprints

In [ ]:
# import os, json, tempfile, requests
# import geopandas as gpd
# from shapely.geometry import Polygon
# import folium


r = requests.get(FOOTPRINTS_URL, params=params)
r.raise_for_status()

with open("footprints.geojson", "wb") as f:
    f.write(r.content)

gdf = gpd.read_file("footprints.geojson")

# normalize BBL
gdf = gdf.rename(columns={"base_bbl": "bbl"})
gdf["bbl"] = gdf["bbl"].astype(str).str.zfill(10)

print("Footprints:", len(gdf))


### Define Polygon (DEBUG ONLY)


In [ ]:
polygon_coords = [
    (-73.99075, 40.73474), #A: 14th & Broadway 
    ( -73.97209, 40.72689), #B: 14th and FDR 
    (-73.9777,  40.71314), #C: Grand & FDR 
    (-73.98249,  40.71468), #D: Grand and E Broadway  
    (-73.99411, 40.71371), # E : E broadway and Manhattan Bridge  
    (-73.99484, 40.7158), #F: Canal and Manhattan bridge
    (-73.9986, 40.71711),  # G: Canal and Mulberry  
    (-74.00187, 40.71939),  # H: Canal and Broadway  
    (-73.99143, 40.73179),  # I: Broadway and 10th 
     (-73.99075, 40.73474)  #A: 14th & Broadway, close polygon
]


# reading 
pv_fp = gpd.read_file("pv_canopy_footprints.geojson")

pv_centroids = gpd.read_file("pv_canopy_centroids.geojson")

poly = Polygon(polygon_coords)

poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")


# clipping
pv_fp_poly = gpd.clip(pv_fp, poly_gdf)
buildings_poly = gpd.clip(gdf, poly_gdf)
pv_centroids_poly = gpd.clip(pv_centroids, poly_gdf)



bbox_geom = box(W_LON, S_LAT, E_LON, N_LAT)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

# clean data
pv_fp = pv_fp[[
    "bbl",
    "pv_kw_dc",
    "annual_kwh",
    "geometry"
]]

### Generate PV Data from Footprints

In [13]:
subprocess.run(["python", "pv_from_footprints.py"], check=True)

PV artifacts written.


CompletedProcess(args=['python', 'pv_from_footprints.py'], returncode=0)

### Load PV Outputs

In [14]:
pv_fp = gpd.read_file("pv_canopy_footprints.geojson")

pv_centroids = gpd.read_file("pv_canopy_centroids.geojson")

print("PV footprints:", len(pv_fp))
print("PV centroids:", len(pv_centroids))

PV footprints: 843
PV centroids: 843


### Build Single Map

In [15]:
m = folium.Map(location=[(N_LAT+S_LAT)/2, (E_LON+W_LON)/2], zoom_start=14)

In [ ]:

# print("FINAL COLUMNS:", pv_fp_poly.columns.tolist())


# --- helper: make all folium-safe ---
def make_json_safe(gdf):
    gdf = gdf.copy()
    for col in gdf.columns:
        if col != "geometry":
            gdf[col] = gdf[col].astype(str)
    return gdf


# --- clean ALL layers ---
bbox_safe = make_json_safe(bbox_gdf)
poly_safe = make_json_safe(poly_gdf) #des this ned to be pv fp poly 
buildings_safe = make_json_safe(buildings_poly)

pv_fp_poly_safe = make_json_safe(pv_fp_poly)
pv_centroids_poly_safe = make_json_safe(pv_centroids_poly)

print("FINAL COLUMNS:", pv_fp_poly.columns.tolist())
print("FINAL SAFE COLUMNS:", pv_fp_poly_safe.columns.tolist())


# --- Bounding Box ---
folium.GeoJson(
    bbox_safe,
    name="Bounding Box Perimeter",
    style_function=lambda x: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)



# --- Polygon AOI ---
folium.GeoJson(
    poly_safe,
    name="Polygon AOI Perimeter",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)


# --- Building Footprints (AOI) ---
folium.GeoJson(
    buildings_safe,
    name="Building Footprints (All)",
    style_function=lambda x: {
        "fill": "#9ecae1",
        "stroke": "#6baed6",
        "weight": 1,
        "fillOpacity": 0.2
    }
).add_to(m)


# --- PV Footprints (with tooltip) ---
folium.GeoJson(
    pv_fp_poly_safe,
    name="PV Canopy Footprints",
    tooltip=folium.GeoJsonTooltip(
        fields=["address", "bldgclass", "pv_kw_dc", "annual_kwh"],
        aliases=["Address", "Class", "kW", "Annual kWh"],
        sticky=True
    ),
    style_function=lambda x: {
         "fillColor": "#ffb347",
    "color": "#f59f00",
    "weight": 1,
    "fillOpacity": 0.6
    }
).add_to(m)


# --- PV Centroids (clustered + toggleable) ---
# --- PV Centroids (clustered + toggleable) ---
cluster_group = folium.FeatureGroup(name="PV Sites (Centroids)", show=True)
cluster = MarkerCluster().add_to(cluster_group)

for r in pv_centroids_poly.itertuples(index=False):
    lat, lon = r.geometry.y, r.geometry.x

    addr = getattr(r, "address", None) or "(address unavailable)"
    bbl  = getattr(r, "bbl", None) or "(BBL unavailable)"
    pvkw = getattr(r, "pv_kw_dc", None)
    pshs = getattr(r, "psh_daily_kwh", None)

    pvkw_txt = f"{float(pvkw):.1f} kW" if pvkw not in [None, "nan"] else "n/a"
    psh_txt  = f"{float(pshs):.2f} PSH" if pshs not in [None, "nan"] else "n/a"

    html = (
        f"<b>Address:</b> {addr}<br/>"
        f"<b>BBL:</b> {bbl}<br/>"
        f"<b>PV size:</b> {pvkw_txt}<br/>"
        f"<b>PSH:</b> {psh_txt}"
    )

    folium.Marker(
        [lat, lon],
        tooltip=addr,
        popup=folium.Popup(html, max_width=320)
    ).add_to(cluster)

cluster_group.add_to(m)

# --- Layer control + save ---
folium.LayerControl().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

import os
os.makedirs("maps", exist_ok=True)
OUT_HTML = "maps/solar_map.html"


m.save(OUT_HTML)
print("Saved", OUT_HTML)

FINAL COLUMNS: ['name', 'bbl', 'shape_area', 'construction_year', 'ground_elevation', 'mappluto_bbl', 'objectid', 'feature_code', 'shape_length', 'height_roof', 'last_status_type', 'bin', 'last_edited_date', 'geom_source', 'doitt_id', 'pv_kw_dc', 'annual_kwh', 'avg_daily_kwh', 'psh_daily_kwh', 'canopy_area_m2', 'address_x', 'bldgclass_x', 'geometry', 'address_y', 'bldgclass_y']
FINAL SAFE COLUMNS: ['name', 'bbl', 'shape_area', 'construction_year', 'ground_elevation', 'mappluto_bbl', 'objectid', 'feature_code', 'shape_length', 'height_roof', 'last_status_type', 'bin', 'last_edited_date', 'geom_source', 'doitt_id', 'pv_kw_dc', 'annual_kwh', 'avg_daily_kwh', 'psh_daily_kwh', 'canopy_area_m2', 'address_x', 'bldgclass_x', 'geometry', 'address_y', 'bldgclass_y']


AssertionError: The field address is not available in the data. Choose from: ('name', 'bbl', 'shape_area', 'construction_year', 'ground_elevation', 'mappluto_bbl', 'objectid', 'feature_code', 'shape_length', 'height_roof', 'last_status_type', 'bin', 'last_edited_date', 'geom_source', 'doitt_id', 'pv_kw_dc', 'annual_kwh', 'avg_daily_kwh', 'psh_daily_kwh', 'canopy_area_m2', 'address_x', 'bldgclass_x', 'address_y', 'bldgclass_y').